# 🏦 FinGuard AI — Feature Engineering

### AI-Powered Financial Fraud Detection & Risk Intelligence Platform

**Notebook:** 04 — Feature Engineering  
**Project:** FinGuard AI  
**Dataset:** Credit Card Transactions  
**Stage:** Behavioral & Model-Ready Feature Construction

---

#### 1. Objective

This notebook creates additional fraud-oriented features from the preprocessed credit-card transaction data.

The feature engineering stage focuses on:

- Transaction amount behavior
- Temporal transaction patterns
- Amount magnitude
- Statistical behavior of PCA-transformed transaction features
- Risk-oriented deviation features
- Model-ready feature generation

**Important:** The target variable `Class` is never used to create input features.

The original raw dataset remains unchanged.


#### 2. Feature Engineering Strategy

The original dataset contains:

- `Time`
- `V1` to `V28`
- `Amount`
- `Class`

Because the dataset does not contain a card/customer/account identifier, true per-card frequency or per-user velocity features cannot be calculated without inventing information.

Therefore, this notebook uses **dataset-supported behavioral features** rather than fabricated identity-level features.


In [1]:
# ============================================================
# FinGuard AI — Feature Engineering Environment Setup
# ============================================================

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)

print("FinGuard AI — Feature Engineering")
print("=" * 70)
print(f"Python Version : {sys.version.split()[0]}")
print(f"NumPy Version  : {np.__version__}")
print(f"Pandas Version : {pd.__version__}")


FinGuard AI — Feature Engineering
Python Version : 3.11.15
NumPy Version  : 2.4.6
Pandas Version : 3.0.5


#### 3. Project Paths

In [2]:
# ============================================================
# Project Directory Configuration
# ============================================================

PROJECT_ROOT = Path.cwd().parent

DATASET_DIR = PROJECT_ROOT / "Dataset"
PROCESSED_DIR = DATASET_DIR / "Processed"
MODELS_DIR = PROJECT_ROOT / "Models"
REPORTS_DIR = PROJECT_ROOT / "Reports"
FIGURES_DIR = REPORTS_DIR / "EDA" / "Figures"

TRAIN_PATH = PROCESSED_DIR / "train_processed.csv"
TRAIN_BALANCED_PATH = PROCESSED_DIR / "train_balanced_smote.csv"
TEST_PATH = PROCESSED_DIR / "test_processed.csv"

SCALER_PATH = MODELS_DIR / "standard_scaler.joblib"
METADATA_PATH = MODELS_DIR / "preprocessing_metadata.joblib"

FEATURE_TRAIN_PATH = PROCESSED_DIR / "feature_engineered_train.csv"
FEATURE_TRAIN_BALANCED_PATH = PROCESSED_DIR / "feature_engineered_train_balanced.csv"
FEATURE_TEST_PATH = PROCESSED_DIR / "feature_engineered_test.csv"
FEATURE_METADATA_PATH = MODELS_DIR / "feature_engineering_metadata.joblib"

print("Project Paths")
print("=" * 70)
print(f"Project Root        : {PROJECT_ROOT}")
print(f"Processed Dataset   : {PROCESSED_DIR}")
print(f"Training Dataset    : {TRAIN_PATH}")
print(f"Balanced Training   : {TRAIN_BALANCED_PATH}")
print(f"Testing Dataset     : {TEST_PATH}")


Project Paths
Project Root        : C:\Users\viqua\Desktop\FinGuard AI
Processed Dataset   : C:\Users\viqua\Desktop\FinGuard AI\Dataset\Processed
Training Dataset    : C:\Users\viqua\Desktop\FinGuard AI\Dataset\Processed\train_processed.csv
Balanced Training   : C:\Users\viqua\Desktop\FinGuard AI\Dataset\Processed\train_balanced_smote.csv
Testing Dataset     : C:\Users\viqua\Desktop\FinGuard AI\Dataset\Processed\test_processed.csv


#### 4. Validate Required Inputs

In [3]:
# ============================================================
# Input Validation
# ============================================================

required_files = [
    TRAIN_PATH,
    TRAIN_BALANCED_PATH,
    TEST_PATH,
    SCALER_PATH,
    METADATA_PATH
]

print("Required Input Validation")
print("=" * 70)

for file_path in required_files:
    status = "✓ EXISTS" if file_path.exists() else "✗ MISSING"
    print(f"{status:<12} {file_path}")

missing_files = [p for p in required_files if not p.exists()]

if missing_files:
    raise FileNotFoundError(
        "Required preprocessing artifacts are missing. "
        "Complete Notebook 03 before running Notebook 04."
    )

print("=" * 70)
print("✓ All required preprocessing artifacts are available")


Required Input Validation
✓ EXISTS     C:\Users\viqua\Desktop\FinGuard AI\Dataset\Processed\train_processed.csv
✓ EXISTS     C:\Users\viqua\Desktop\FinGuard AI\Dataset\Processed\train_balanced_smote.csv
✓ EXISTS     C:\Users\viqua\Desktop\FinGuard AI\Dataset\Processed\test_processed.csv
✓ EXISTS     C:\Users\viqua\Desktop\FinGuard AI\Models\standard_scaler.joblib
✓ EXISTS     C:\Users\viqua\Desktop\FinGuard AI\Models\preprocessing_metadata.joblib
✓ All required preprocessing artifacts are available


#### 5. Load Preprocessed Data

Three datasets are loaded:

1. Training data before SMOTE
2. Balanced training data after SMOTE
3. Test data

The test set remains untouched by SMOTE.


In [4]:
# ============================================================
# Load Preprocessed Datasets
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
train_balanced_df = pd.read_csv(TRAIN_BALANCED_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Dataset Loading")
print("=" * 70)
print(f"Training Samples          : {len(train_df):,}")
print(f"Balanced Training Samples : {len(train_balanced_df):,}")
print(f"Testing Samples           : {len(test_df):,}")
print(f"Training Columns          : {train_df.shape[1]}")
print(f"Testing Columns           : {test_df.shape[1]}")


Dataset Loading
Training Samples          : 226,980
Balanced Training Samples : 453,204
Testing Samples           : 56,746
Training Columns          : 31
Testing Columns           : 31


#### 6. Load Preprocessing Metadata

Notebook 03 stored the scaler and feature metadata. These artifacts allow the original `Time` and `Amount` scale to be reconstructed before creating meaningful amount and temporal features.


In [5]:
# ============================================================
# Load Scaler and Metadata
# ============================================================

scaler = joblib.load(SCALER_PATH)
preprocessing_metadata = joblib.load(METADATA_PATH)

print("Preprocessing Artifacts")
print("=" * 70)
print(f"Scaler Type : {type(scaler).__name__}")
print(f"Metadata Type : {type(preprocessing_metadata).__name__}")

if isinstance(preprocessing_metadata, dict):
    print("Metadata Keys:")
    for key in preprocessing_metadata.keys():
        print(f"  - {key}")


Preprocessing Artifacts
Scaler Type : StandardScaler
Metadata Type : dict
Metadata Keys:
  - scale_columns
  - target_column
  - random_state
  - test_size
  - duplicate_rows_removed
  - smote_applied
  - dataset_file


#### 7. Identify Feature Columns and Target

In [6]:
# ============================================================
# Feature / Target Identification
# ============================================================

TARGET = "Class"

if TARGET not in train_df.columns:
    raise KeyError("Target column 'Class' was not found.")

feature_columns = [c for c in train_df.columns if c != TARGET]

v_columns = [c for c in feature_columns if c.startswith("V") and c[1:].isdigit()]
has_time = "Time" in feature_columns
has_amount = "Amount" in feature_columns

print("Feature Structure")
print("=" * 70)
print(f"Target Column : {TARGET}")
print(f"Feature Count : {len(feature_columns)}")
print(f"V Features    : {len(v_columns)}")
print(f"Time Present  : {has_time}")
print(f"Amount Present: {has_amount}")

if not has_time or not has_amount:
    raise KeyError("Expected Time and Amount columns were not found.")

print("=" * 70)
print("✓ Feature structure validated")


Feature Structure
Target Column : Class
Feature Count : 30
V Features    : 28
Time Present  : True
Amount Present: True
✓ Feature structure validated


#### 8. Reconstruct Original Time and Amount Scale

Notebook 03 standardized the numerical features. For meaningful feature engineering, `Time` and `Amount` are reconstructed using the fitted `StandardScaler`.

No target information is used.


In [7]:
# ============================================================
# Recover Original Time / Amount Values
# ============================================================

scaler_features = list(getattr(scaler, "feature_names_in_", feature_columns))

if "Time" not in scaler_features or "Amount" not in scaler_features:
    raise KeyError("Time/Amount are not available in the saved scaler metadata.")

time_index = scaler_features.index("Time")
amount_index = scaler_features.index("Amount")

time_mean = float(scaler.mean_[time_index])
time_scale = float(scaler.scale_[time_index])

amount_mean = float(scaler.mean_[amount_index])
amount_scale = float(scaler.scale_[amount_index])

def recover_original_values(df):
    result = df.copy()
    result["_Time_Original"] = result["Time"] * time_scale + time_mean
    result["_Amount_Original"] = result["Amount"] * amount_scale + amount_mean
    return result

train_work = recover_original_values(train_df)
train_balanced_work = recover_original_values(train_balanced_df)
test_work = recover_original_values(test_df)

print("Original Scale Reconstruction")
print("=" * 70)
print(f"Time Mean Used   : {time_mean:.6f}")
print(f"Time Scale Used  : {time_scale:.6f}")
print(f"Amount Mean Used : {amount_mean:.6f}")
print(f"Amount Scale Used: {amount_scale:.6f}")
print("✓ Time and Amount reconstructed")


Original Scale Reconstruction
Time Mean Used   : 94900.224390
Time Scale Used  : 47488.109673
Amount Mean Used : 88.387124
Amount Scale Used: 245.766505
✓ Time and Amount reconstructed


#### 9. Transaction Amount Features

Amount-related features capture transaction magnitude and reduce the effect of extreme skew.

Generated features:

- `Amount_Log1p`
- `Amount_Sqrt`
- `Amount_Squared`
- `Amount_Is_Zero`
- `Amount_Is_High`


In [8]:
# ============================================================
# Transaction Amount Feature Engineering
# ============================================================

def add_amount_features(df):
    result = df.copy()

    amount = result["_Amount_Original"].clip(lower=0)

    result["Amount_Log1p"] = np.log1p(amount)
    result["Amount_Sqrt"] = np.sqrt(amount)
    result["Amount_Squared"] = amount ** 2
    result["Amount_Is_Zero"] = (amount == 0).astype("int8")

    high_threshold = amount.quantile(0.95)
    result["Amount_Is_High"] = (amount >= high_threshold).astype("int8")

    return result

train_work = add_amount_features(train_work)
train_balanced_work = add_amount_features(train_balanced_work)
test_work = add_amount_features(test_work)

print("Amount Features Created")
print("=" * 70)
print("✓ Amount_Log1p")
print("✓ Amount_Sqrt")
print("✓ Amount_Squared")
print("✓ Amount_Is_Zero")
print("✓ Amount_Is_High")


Amount Features Created
✓ Amount_Log1p
✓ Amount_Sqrt
✓ Amount_Squared
✓ Amount_Is_Zero
✓ Amount_Is_High


#### 10. Temporal Features

`Time` represents elapsed seconds in the original credit-card dataset.

The following features model daily transaction timing:

- Day index
- Hour within day
- Minute within hour
- Cyclical hour encoding
- Cyclical day-position encoding


In [9]:
# ============================================================
# Temporal Feature Engineering
# ============================================================

SECONDS_PER_DAY = 24 * 60 * 60
SECONDS_PER_HOUR = 60 * 60
SECONDS_PER_MINUTE = 60

def add_temporal_features(df):
    result = df.copy()

    time_value = result["_Time_Original"].clip(lower=0)

    result["Time_Day_Index"] = np.floor(time_value / SECONDS_PER_DAY).astype("int32")

    seconds_in_day = time_value % SECONDS_PER_DAY
    hour = np.floor(seconds_in_day / SECONDS_PER_HOUR)
    minute = np.floor((seconds_in_day % SECONDS_PER_HOUR) / SECONDS_PER_MINUTE)

    result["Time_Hour"] = hour.astype("int8")
    result["Time_Minute"] = minute.astype("int8")

    result["Time_Hour_Sin"] = np.sin(2 * np.pi * hour / 24)
    result["Time_Hour_Cos"] = np.cos(2 * np.pi * hour / 24)

    result["Time_Day_Sin"] = np.sin(2 * np.pi * seconds_in_day / SECONDS_PER_DAY)
    result["Time_Day_Cos"] = np.cos(2 * np.pi * seconds_in_day / SECONDS_PER_DAY)

    return result

train_work = add_temporal_features(train_work)
train_balanced_work = add_temporal_features(train_balanced_work)
test_work = add_temporal_features(test_work)

print("Temporal Features Created")
print("=" * 70)
for name in [
    "Time_Day_Index", "Time_Hour", "Time_Minute",
    "Time_Hour_Sin", "Time_Hour_Cos",
    "Time_Day_Sin", "Time_Day_Cos"
]:
    print(f"✓ {name}")


Temporal Features Created
✓ Time_Day_Index
✓ Time_Hour
✓ Time_Minute
✓ Time_Hour_Sin
✓ Time_Hour_Cos
✓ Time_Day_Sin
✓ Time_Day_Cos


#### 11. Transaction Feature Statistics

The anonymized `V1`–`V28` variables are transformed transaction components. Their collective behavior can be summarized without changing the original variables.


In [10]:
# ============================================================
# V1-V28 Statistical Features
# ============================================================

def add_v_statistical_features(df):
    result = df.copy()

    v = result[v_columns]

    result["V_Abs_Mean"] = v.abs().mean(axis=1)
    result["V_Abs_Max"] = v.abs().max(axis=1)
    result["V_Abs_Std"] = v.abs().std(axis=1).fillna(0)
    result["V_L2_Norm"] = np.sqrt((v ** 2).sum(axis=1))
    result["V_Positive_Count"] = (v > 0).sum(axis=1).astype("int8")
    result["V_Negative_Count"] = (v < 0).sum(axis=1).astype("int8")
    result["V_Extreme_Count"] = (v.abs() > 3).sum(axis=1).astype("int8")

    return result

train_work = add_v_statistical_features(train_work)
train_balanced_work = add_v_statistical_features(train_balanced_work)
test_work = add_v_statistical_features(test_work)

print("V-Feature Statistical Features Created")
print("=" * 70)
for name in [
    "V_Abs_Mean", "V_Abs_Max", "V_Abs_Std", "V_L2_Norm",
    "V_Positive_Count", "V_Negative_Count", "V_Extreme_Count"
]:
    print(f"✓ {name}")


V-Feature Statistical Features Created
✓ V_Abs_Mean
✓ V_Abs_Max
✓ V_Abs_Std
✓ V_L2_Norm
✓ V_Positive_Count
✓ V_Negative_Count
✓ V_Extreme_Count


#### 12. Risk-Oriented Deviation Features

Large deviations across anonymized transaction components can be useful for fraud-risk modeling.

These features summarize the intensity and number of extreme component values.


In [11]:
# ============================================================
# Risk-Oriented Behavioral Features
# ============================================================

def add_risk_features(df):
    result = df.copy()

    v = result[v_columns]

    result["Risk_Deviation_Score"] = v.abs().mean(axis=1)
    result["Risk_Max_Deviation"] = v.abs().max(axis=1)
    result["Risk_Extreme_Ratio"] = (v.abs() > 3).mean(axis=1)
    result["Risk_Severe_Ratio"] = (v.abs() > 5).mean(axis=1)

    return result

train_work = add_risk_features(train_work)
train_balanced_work = add_risk_features(train_balanced_work)
test_work = add_risk_features(test_work)

print("Risk-Oriented Features Created")
print("=" * 70)
print("✓ Risk_Deviation_Score")
print("✓ Risk_Max_Deviation")
print("✓ Risk_Extreme_Ratio")
print("✓ Risk_Severe_Ratio")


Risk-Oriented Features Created
✓ Risk_Deviation_Score
✓ Risk_Max_Deviation
✓ Risk_Extreme_Ratio
✓ Risk_Severe_Ratio


#### 13. Remove Temporary Reconstruction Columns

The reconstructed helper columns are only intermediate calculations and are not retained in the final model-ready datasets.


In [12]:
# ============================================================
# Remove Temporary Columns
# ============================================================

TEMP_COLUMNS = ["_Time_Original", "_Amount_Original"]

for df_name, frame in [
    ("train", train_work),
    ("train_balanced", train_balanced_work),
    ("test", test_work)
]:
    frame.drop(columns=TEMP_COLUMNS, inplace=True, errors="ignore")

print("✓ Temporary reconstruction columns removed")


✓ Temporary reconstruction columns removed


#### 14. Validate Feature Engineering Output

In [13]:
# ============================================================
# Feature Engineering Validation
# ============================================================

for name, frame in [
    ("Training", train_work),
    ("Balanced Training", train_balanced_work),
    ("Testing", test_work)
]:
    print(f"\n{name}")
    print("-" * 70)
    print(f"Rows    : {len(frame):,}")
    print(f"Columns : {frame.shape[1]:,}")
    print(f"Missing : {int(frame.isna().sum().sum()):,}")
    print(f"Inf     : {int(np.isinf(frame.select_dtypes(include=np.number)).sum().sum()):,}")

if TARGET not in train_work.columns:
    raise KeyError("Target column was lost during feature engineering.")

print("\n✓ Feature engineering validation passed")



Training
----------------------------------------------------------------------
Rows    : 226,980
Columns : 54
Missing : 0
Inf     : 0

Balanced Training
----------------------------------------------------------------------
Rows    : 453,204
Columns : 54
Missing : 0
Inf     : 0

Testing
----------------------------------------------------------------------
Rows    : 56,746
Columns : 54
Missing : 0
Inf     : 0

✓ Feature engineering validation passed


#### 15. Verify Target Isolation

The target `Class` is retained only as the prediction label. No engineered feature is derived from it.


In [14]:
# ============================================================
# Target Isolation Check
# ============================================================

engineered_feature_columns = [
    c for c in train_work.columns
    if c != TARGET
]

for forbidden in ["Class", "class", "Fraud", "Target"]:
    if forbidden in engineered_feature_columns:
        raise ValueError(f"Target leakage detected through feature: {forbidden}")

print("Target Isolation")
print("=" * 70)
print(f"Target Column        : {TARGET}")
print(f"Engineered Features  : {len(engineered_feature_columns)}")
print("✓ No target-derived feature detected")


Target Isolation
Target Column        : Class
Engineered Features  : 53
✓ No target-derived feature detected


#### 16. Feature Inventory

In [15]:
# ============================================================
# Feature Inventory
# ============================================================

original_features = feature_columns
new_features = [
    c for c in engineered_feature_columns
    if c not in original_features
]

feature_inventory = pd.DataFrame({
    "Feature": new_features,
    "Feature_Group": [
        (
            "Amount"
            if c.startswith("Amount_")
            else "Temporal"
            if c.startswith("Time_")
            else "V_Statistics"
            if c.startswith("V_")
            else "Risk"
        )
        for c in new_features
    ]
})

display(feature_inventory)

print("=" * 70)
print(f"Original Features : {len(original_features)}")
print(f"New Features      : {len(new_features)}")
print(f"Final Features    : {len(engineered_feature_columns)}")


,Feature,Feature_Group
0,Amount_Log1p,Amount
1,Amount_Sqrt,Amount
2,Amount_Squared,Amount
3,Amount_Is_Zero,Amount
4,Amount_Is_High,Amount
5,Time_Day_Index,Temporal
6,Time_Hour,Temporal
7,Time_Minute,Temporal
8,Time_Hour_Sin,Temporal
9,Time_Hour_Cos,Temporal


Original Features : 30
New Features      : 23
Final Features    : 53


#### 17. Engineered Feature Preview

In [16]:
# ============================================================
# Preview Engineered Features
# ============================================================

preview_columns = new_features[:25]

display(train_work[preview_columns].head(10))

print("✓ Engineered feature preview generated")


,Amount_Log1p,Amount_Sqrt,Amount_Squared,Amount_Is_Zero,Amount_Is_High,Time_Day_Index,Time_Hour,Time_Minute,Time_Hour_Sin,Time_Hour_Cos,Time_Day_Sin,Time_Day_Cos,V_Abs_Mean,V_Abs_Max,V_Abs_Std,V_L2_Norm,V_Positive_Count,V_Negative_Count,V_Extreme_Count,Risk_Deviation_Score,Risk_Max_Deviation,Risk_Extreme_Ratio,Risk_Severe_Ratio
0,3.496508,5.656854,1024.0000,0,0,1,16,9,-0.866025,-5.000000e-01,-0.885292,-0.465035,0.858682,3.565492,0.913525,6.570969,11,17,1,0.858682,3.565492,0.035714,0.000000
1,2.078191,2.643861,48.8601,0,0,0,22,25,-0.500000,8.660254e-01,-0.401681,0.915779,0.634595,1.762389,0.507786,4.270568,13,15,0,0.634595,1.762389,0.000000,0.000000
2,2.769459,3.866523,223.5025,0,0,1,11,18,0.258819,-9.659258e-01,0.179661,-0.983729,0.620295,2.087997,0.617911,4.591559,17,11,0,0.620295,2.087997,0.000000,0.000000
3,2.906901,4.159327,299.2900,0,0,1,1,22,0.258819,9.659258e-01,0.351706,0.936111,0.904225,3.880231,0.885021,6.636381,18,10,1,0.904225,3.880231,0.035714,0.000000
4,3.218876,4.898979,576.0000,0,0,0,8,12,0.866025,-5.000000e-01,0.838591,-0.544761,0.673056,2.094285,0.575181,4.649367,13,15,0,0.673056,2.094285,0.000000,0.000000
5,2.301585,2.998333,80.8201,0,0,0,16,49,-0.866025,-5.000000e-01,-0.953410,-0.301677,1.578944,7.406095,1.884045,12.870341,15,13,5,1.578944,7.406095,0.178571,0.071429
6,3.432373,5.472659,897.0025,0,0,0,10,34,0.500000,-8.660254e-01,0.366366,-0.930471,0.399480,1.285426,0.374431,2.872930,13,15,0,0.399480,1.285426,0.000000,0.000000
7,3.699819,6.280127,1555.5136,0,0,1,22,51,-0.500000,8.660254e-01,-0.296333,0.955085,0.709978,3.788902,0.850852,5.801771,10,18,1,0.709978,3.788902,0.035714,0.000000
8,1.811562,2.262742,26.2144,0,0,1,19,9,-0.965926,2.588190e-01,-0.954132,0.299388,0.477480,2.012594,0.487968,3.579481,16,12,0,0.477480,2.012594,0.000000,0.000000
9,0.570980,0.877496,0.5929,0,0,1,18,42,-1.000000,-1.836970e-16,-0.982545,0.186024,0.781635,2.907999,0.800056,5.864223,17,11,0,0.781635,2.907999,0.000000,0.000000


✓ Engineered feature preview generated


#### 18. Engineered Feature Summary Statistics

In [17]:
# ============================================================
# Feature Statistics
# ============================================================

display(
    train_work[new_features]
    .describe()
    .T
    .round(4)
)

print("✓ Feature statistics generated")


,count,mean,std,min,25%,50%,75%,max
Amount_Log1p,226980.0,3.1561,1.656200e+00,0.0000,1.9006,3.1390,4.3675,9.886200e+00
Amount_Sqrt,226980.0,6.6934,6.602000e+00,0.0000,2.3854,4.6989,8.8233,1.402017e+02
Amount_Squared,226980.0,68213.4585,1.436983e+06,0.0000,32.3761,487.5264,6060.6225,3.863792e+08
Amount_Is_Zero,226980.0,0.0000,0.000000e+00,0.0000,0.0000,0.0000,0.0000,0.000000e+00
Amount_Is_High,226980.0,0.0500,2.180000e-01,0.0000,0.0000,0.0000,0.0000,1.000000e+00
Time_Day_Index,226980.0,0.4923,4.999000e-01,0.0000,0.0000,0.0000,1.0000,1.000000e+00
Time_Hour,226980.0,14.0544,5.828900e+00,0.0000,10.0000,15.0000,19.0000,2.300000e+01
Time_Minute,226980.0,29.0119,1.741150e+01,0.0000,14.0000,29.0000,44.0000,5.900000e+01
Time_Hour_Sin,226980.0,-0.2447,6.504000e-01,-1.0000,-0.8660,-0.5000,0.2588,1.000000e+00
Time_Hour_Cos,226980.0,-0.1750,6.975000e-01,-1.0000,-0.8660,-0.2588,0.5000,1.000000e+00


✓ Feature statistics generated


#### 19. Check Feature Variance

Constant features provide no useful information and are flagged for review.


In [18]:
# ============================================================
# Constant / Near-Constant Feature Check
# ============================================================

variance_table = pd.DataFrame({
    "Feature": engineered_feature_columns,
    "Variance": [
        float(train_work[c].var()) if pd.api.types.is_numeric_dtype(train_work[c]) else np.nan
        for c in engineered_feature_columns
    ]
})

constant_features = variance_table[
    variance_table["Variance"].fillna(1) == 0
]["Feature"].tolist()

print("Feature Variance Check")
print("=" * 70)

if constant_features:
    print("Constant Features:")
    for feature in constant_features:
        print(f"  - {feature}")
else:
    print("✓ No constant numeric features detected")


Feature Variance Check
Constant Features:
  - Amount_Is_Zero


#### 20. Prepare Final Model-Ready Feature Tables

The target is kept as `Class`. Feature engineering is applied consistently to:

- Original training data
- SMOTE-balanced training data
- Unseen test data


In [19]:
# ============================================================
# Final Feature-Engineered Tables
# ============================================================

feature_train_df = train_work.copy()
feature_train_balanced_df = train_balanced_work.copy()
feature_test_df = test_work.copy()

print("Final Feature-Engineered Tables")
print("=" * 70)
print(f"Training          : {feature_train_df.shape}")
print(f"Balanced Training : {feature_train_balanced_df.shape}")
print(f"Testing           : {feature_test_df.shape}")


Final Feature-Engineered Tables
Training          : (226980, 54)
Balanced Training : (453204, 54)
Testing           : (56746, 54)


#### 21. Save Feature-Engineered Datasets

In [20]:
# ============================================================
# Save Feature-Engineered Datasets
# ============================================================

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

feature_train_df.to_csv(FEATURE_TRAIN_PATH, index=False)
feature_train_balanced_df.to_csv(FEATURE_TRAIN_BALANCED_PATH, index=False)
feature_test_df.to_csv(FEATURE_TEST_PATH, index=False)

print("Feature-Engineered Datasets Saved")
print("=" * 70)
print(f"✓ {FEATURE_TRAIN_PATH}")
print(f"✓ {FEATURE_TRAIN_BALANCED_PATH}")
print(f"✓ {FEATURE_TEST_PATH}")


Feature-Engineered Datasets Saved
✓ C:\Users\viqua\Desktop\FinGuard AI\Dataset\Processed\feature_engineered_train.csv
✓ C:\Users\viqua\Desktop\FinGuard AI\Dataset\Processed\feature_engineered_train_balanced.csv
✓ C:\Users\viqua\Desktop\FinGuard AI\Dataset\Processed\feature_engineered_test.csv


#### 22. Save Feature Engineering Metadata

In [21]:
# ============================================================
# Save Feature Engineering Metadata
# ============================================================

feature_metadata = {
    "target_column": TARGET,
    "original_feature_count": len(original_features),
    "new_feature_count": len(new_features),
    "final_feature_count": len(engineered_feature_columns),
    "original_features": original_features,
    "engineered_features": new_features,
    "final_features": engineered_feature_columns,
    "feature_groups": feature_inventory.to_dict(orient="records"),
    "smote_applied_before_feature_engineering": True,
    "target_used_for_feature_creation": False
}

joblib.dump(feature_metadata, FEATURE_METADATA_PATH)

print("✓ Feature engineering metadata saved")
print(f"Location: {FEATURE_METADATA_PATH}")


✓ Feature engineering metadata saved
Location: C:\Users\viqua\Desktop\FinGuard AI\Models\feature_engineering_metadata.joblib


#### 23. Final Artifact Validation

In [22]:
# ============================================================
# Final Artifact Validation
# ============================================================

generated_files = [
    FEATURE_TRAIN_PATH,
    FEATURE_TRAIN_BALANCED_PATH,
    FEATURE_TEST_PATH,
    FEATURE_METADATA_PATH
]

print("Generated Feature Engineering Artifacts")
print("=" * 75)

for file_path in generated_files:
    status = "✓ EXISTS" if file_path.exists() else "✗ MISSING"
    size_mb = file_path.stat().st_size / (1024 ** 2) if file_path.exists() else 0
    print(f"{status:<12} {file_path}  ({size_mb:.2f} MB)")

if not all(p.exists() for p in generated_files):
    raise RuntimeError("One or more feature engineering artifacts were not created.")

print("=" * 75)
print("✓ Artifact validation passed")


Generated Feature Engineering Artifacts
✓ EXISTS     C:\Users\viqua\Desktop\FinGuard AI\Dataset\Processed\feature_engineered_train.csv  (178.46 MB)
✓ EXISTS     C:\Users\viqua\Desktop\FinGuard AI\Dataset\Processed\feature_engineered_train_balanced.csv  (365.89 MB)
✓ EXISTS     C:\Users\viqua\Desktop\FinGuard AI\Dataset\Processed\feature_engineered_test.csv  (44.61 MB)
✓ EXISTS     C:\Users\viqua\Desktop\FinGuard AI\Models\feature_engineering_metadata.joblib  (0.00 MB)
✓ Artifact validation passed


#### 24. Modeling Note

For downstream modeling:

- Use `feature_engineered_train_balanced.csv` for models that require balanced training data.
- Use `feature_engineered_train.csv` when evaluating the effect of imbalance handling.
- Use `feature_engineered_test.csv` only for final model evaluation.
- Do not apply SMOTE to the test dataset.
- Do not use `Class` as an input feature.
- Preserve the same feature order between training and testing.


In [23]:
# ============================================================
# Final FinGuard AI Feature Engineering Summary
# ============================================================

print("=" * 75)
print("FIN GUARD AI — FEATURE ENGINEERING SUMMARY")
print("=" * 75)
print(f"✓ Original Features          : {len(original_features)}")
print(f"✓ New Engineered Features    : {len(new_features)}")
print(f"✓ Final Model Features       : {len(engineered_feature_columns)}")
print(f"✓ Training Samples            : {len(feature_train_df):,}")
print(f"✓ Balanced Training Samples   : {len(feature_train_balanced_df):,}")
print(f"✓ Testing Samples             : {len(feature_test_df):,}")
print("✓ Amount Features             : Created")
print("✓ Temporal Features           : Created")
print("✓ V-Statistical Features      : Created")
print("✓ Risk-Oriented Features      : Created")
print("✓ Target Leakage Check        : PASSED")
print("✓ Feature Metadata             : Saved")
print("✓ Feature-Engineered Datasets : Saved")
print("=" * 75)
print("✓ NOTEBOOK 04 FEATURE ENGINEERING COMPLETED")
print("=" * 75)


FIN GUARD AI — FEATURE ENGINEERING SUMMARY
✓ Original Features          : 30
✓ New Engineered Features    : 23
✓ Final Model Features       : 53
✓ Training Samples            : 226,980
✓ Balanced Training Samples   : 453,204
✓ Testing Samples             : 56,746
✓ Amount Features             : Created
✓ Temporal Features           : Created
✓ V-Statistical Features      : Created
✓ Risk-Oriented Features      : Created
✓ Target Leakage Check        : PASSED
✓ Feature Metadata             : Saved
✓ Feature-Engineered Datasets : Saved
✓ NOTEBOOK 04 FEATURE ENGINEERING COMPLETED


#### 25. Next Stage

### Notebook 05 — Model Training

The next notebook will train and compare fraud detection models using the engineered features.

Planned models:

- Logistic Regression
- Random Forest
- XGBoost
- Support Vector Machine
- K-Nearest Neighbors

Evaluation will focus on:

- Precision
- Recall
- F1-score
- ROC-AUC
- PR-AUC
- False Positive Rate
- False Negative Rate

The best model will then be selected for the FinGuard AI fraud-risk pipeline.
